# 04 — End to end: synthesize → evaluate → export

The synthesis and export steps run fully offline. The evaluate step needs a live
Azure model plus the model dependencies, so it is skipped with a message when no
credentials are configured.

## 1. Synthesize — an adversarial testset from the curated bank (offline)

In [ ]:
from llminspector.synthesizer import AdversarialSynthesizer

synth = AdversarialSynthesizer.from_excel(
    "../tests/test_sample/test_adversarialdata.xlsx",
    capability="all",
    sample_size=5,
)
seed_dataset = synth.generate()
print(f"Synthesized {len(seed_dataset.goldens)} adversarial goldens.")
synth.to_pandas().head()

## 2. Your system answers the seeds

Stubbed here so the flow is self-contained; in practice you would call your LLM
application for each seed prompt.

In [ ]:
from llminspector.dataset import EvaluationDataset
from llminspector.test_case import LLMTestCase

answered = EvaluationDataset(
    test_cases=[
        LLMTestCase(input=g.input, actual_output="(stubbed answer)")
        for g in seed_dataset.goldens
    ]
)
len(answered)

## 3. Evaluate — only when a live model is configured

In [ ]:
import os

from llminspector import a_evaluate, reporting

have_creds = all(
    os.getenv(f"LLMINSPECTOR_{k}")
    for k in ("AZURE_ENDPOINT", "API_VERSION", "API_KEY")
)

if have_creds:
    from llminspector.config import AzureSettings
    from llminspector.metrics import AnswerJailbreakMetric, SentimentMetric
    from llminspector.models import AzureOpenAIModel

    model = AzureOpenAIModel(AzureSettings.from_env())
    metrics = [
        SentimentMetric(model, target="actual_output"),
        AnswerJailbreakMetric(model),
    ]
    result = await a_evaluate(answered, metrics)
    display(result.to_pandas())
    print(reporting.summary(result))

    result.to_excel("/tmp/llminspector_e2e_eval.xlsx")
    print("Wrote /tmp/llminspector_e2e_eval.xlsx")
else:
    print("No Azure credentials found — skipping the evaluate step.")

## 4. Export the synthesized seed set (always)

In [ ]:
synth.to_excel("/tmp/llminspector_e2e_seeds.xlsx")
print("Wrote /tmp/llminspector_e2e_seeds.xlsx")